In [54]:
pip install importlib-resources


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
pip install --upgrade tensorflow-datasets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [56]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow_datasets as tfds
import tensorflow as tf 

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam

In [57]:
dataset, info=tfds.load('oxford_iiit_pet',with_info=True,as_supervised=True)

train_ds=dataset['train']
test_ds=dataset['test']

In [58]:
IMG_SIZE=128
BATCH_SIZE=32

In [59]:
def preprocess(image, label):
    image=tf.image.resize(image,(IMG_SIZE, IMG_SIZE))
    image=tf.cast(image, tf.float32)/255.0
    
    return image, label


In [60]:
def augment(image, label):
    image=tf.image.random_flip_left_right(image)
    return image, label

In [61]:
train_ds=(
    train_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .map(augment,num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)

)
test_ds=(
    test_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)

)

In [62]:
base_model=MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128, 128, 3)
)
base_model.trainable= False

In [63]:
model=models.Sequential([
    base_model,

    layers.GlobalAveragePooling2D(),
    layers.Dense(237, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(37, activation='softmax')
])

In [64]:
for x,y in test_ds.take(1):
    print(x.shape)

(32, 128, 128, 3)


In [65]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [67]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10
)

Epoch 1/10
115/115 ━━━━━━━━━━━━━━━━━━━━ 62s 478ms/step - accuracy: 0.5057 - loss: 1.8109 - val_accuracy: 0.7672 - val_loss: 0.7515
Epoch 2/10
115/115 ━━━━━━━━━━━━━━━━━━━━ 55s 475ms/step - accuracy: 0.7568 - loss: 0.7795 - val_accuracy: 0.7850 - val_loss: 0.6740
Epoch 3/10
115/115 ━━━━━━━━━━━━━━━━━━━━ 55s 478ms/step - accuracy: 0.8247 - loss: 0.5495 - val_accuracy: 0.7817 - val_loss: 0.6836
Epoch 4/10
115/115 ━━━━━━━━━━━━━━━━━━━━ 56s 483ms/step - accuracy: 0.8611 - loss: 0.4236 - val_accuracy: 0.7978 - val_loss: 0.6524
Epoch 5/10
115/115 ━━━━━━━━━━━━━━━━━━━━ 57s 492ms/step - accuracy: 0.8859 - loss: 0.3471 - val_accuracy: 0.8021 - val_loss: 0.6539
Epoch 6/10
115/115 ━━━━━━━━━━━━━━━━━━━━ 56s 482ms/step - accuracy: 0.9117 - loss: 0.2748 - val_accuracy: 0.8005 - val_loss: 0.6627
Epoch 7/10
115/115 ━━━━━━━━━━━━━━━━━━━━ 55s 476ms/step - accuracy: 0.9220 - loss: 0.2350 - val_accuracy: 0.8013 - val_loss: 0.6733
Epoch 8/10
115/115 ━━━━━━━━━━━━━━━━━━━━ 55s 478ms/step - accuracy: 0.9361 - loss: 0

In [68]:
loss, acc =model.evaluation(test_ds)
print(acc)

AttributeError: 'Sequential' object has no attribute 'evaluation'